In [0]:
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA default")

PREFIX = "wdatt_movie_"


In [0]:
ratings = spark.table(f"{PREFIX}gold_fact_ratings").select("userId","movieId","rating")
print("ratings rows:", ratings.count())


In [0]:
train, test = ratings.randomSplit([0.8, 0.2], seed=42)

als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    rank=20,
    maxIter=10,
    regParam=0.1,
    coldStartStrategy="drop"
)

model = als.fit(train)

pred = model.transform(test)
rmse = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction").evaluate(pred)
print("RMSE:", rmse)


In [0]:
TOPK = 10

item_f = model.itemFactors.select("id", "features").collect()
item_ids = np.array([r["id"] for r in item_f], dtype=np.int32)
item_vecs = np.array([r["features"] for r in item_f], dtype=np.float32)

schema = T.StructType([
    T.StructField("userId", T.IntegerType(), False),
    T.StructField("movieId", T.IntegerType(), False),
    T.StructField("predicted_rating", T.FloatType(), False),
])

rows = []
n_users = 0

for u in model.userFactors.select("id","features").toLocalIterator():
    n_users += 1
    uid = int(u["id"])
    uvec = np.array(u["features"], dtype=np.float32)
    scores = item_vecs @ uvec

    top_idx = np.argpartition(scores, -TOPK)[-TOPK:]
    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]

    for i in top_idx:
        rows.append((uid, int(item_ids[i]), float(scores[i])))

print("users processed:", n_users, "rows:", len(rows))

serving_flat = spark.createDataFrame(rows, schema=schema)
serving_flat.write.format("delta").mode("overwrite").saveAsTable(f"{PREFIX}serving_user_recommendations_flat")


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA default")
PREFIX = "wdatt_movie_"

movies = spark.table(f"{PREFIX}gold_dim_movies_enriched").select("movieId","title_clean","director","poster_url")

serving_enriched = (
    spark.table(f"{PREFIX}serving_user_recommendations_flat")
    .join(movies, on="movieId", how="left")
)

w = Window.partitionBy("userId").orderBy(F.col("predicted_rating").desc())
serving_enriched = serving_enriched.withColumn("rec_rank", F.row_number().over(w))

# ✅ overwrite + allow schema evolution
(serving_enriched.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")   # safest for full overwrite
 .option("mergeSchema", "true")
 .saveAsTable(f"{PREFIX}serving_user_recommendations_enriched"))

display(serving_enriched.limit(20))


In [0]:
assert spark.table(f"{PREFIX}serving_user_recommendations_enriched").count() > 0
assert spark.table(f"{PREFIX}serving_user_recommendations_enriched").select("userId").distinct().count() > 0
assert spark.table(f"{PREFIX}serving_user_recommendations_enriched").where("rec_rank <= 10").count() > 0
print("✅ SERVING validated")


In [0]:
from pyspark.sql import functions as F

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA default")
P = "wdatt_movie_"

checks = {
  f"{P}bronze_movies": None,
  f"{P}bronze_links": None,
  f"{P}bronze_tags": None,
  f"{P}bronze_scraped_metadata": None,
  f"{P}bronze_ratings_top500": None,
  f"{P}silver_movie_master": None,
  f"{P}gold_dim_movies_enriched": 500,
  f"{P}gold_fact_ratings": None,
  f"{P}gold_dim_users": None,
  f"{P}serving_user_recommendations_enriched": None,
}

for t, expected in checks.items():
    c = spark.table(t).count()
    msg = f"{t}: {c:,}"
    if expected is not None:
        msg += f" (expected {expected})"
        assert c == expected, msg
    print(msg)

# quick sanity on rec ranks
rec = spark.table(f"{P}serving_user_recommendations_enriched")
bad = rec.where((F.col("rec_rank") < 1) | (F.col("rec_rank") > 10)).count()
print("bad rec_rank rows:", bad)
assert bad == 0
print("✅ END-TO-END validated")
